# Assigning child resources

Resources form a tree (see the {doc}`introduction`). You build the tree by assigning a child
resource to a parent resource. This page shows the `assign_child_*` methods:

- {meth}`~pylabrobot.resources.Resource.assign_child_resource` puts a child at a location that you
  give.
- {meth}`~pylabrobot.resources.Resource.assign_child_by_anchor` computes the location from two
  anchor points, one on the parent and one on the child.
- Some resources, such as resource holders, carriers and decks, compute the location for you.

## At a location: `assign_child_resource`

The location of a child is relative to the left front bottom corner of its parent. The x-axis goes
from left to right, the y-axis from front to back, and the z-axis from bottom to top.

In [ ]:
from pylabrobot.resources import Coordinate, Resource

parent = Resource(name="parent", size_x=100, size_y=80, size_z=10)
child = Resource(name="child", size_x=20, size_y=10, size_z=5)

parent.assign_child_resource(child, location=Coordinate(x=10, y=20, z=0))

assert child.parent is parent
assert child.location == Coordinate(x=10, y=20, z=0)

{meth}`~pylabrobot.resources.Resource.get_absolute_location` gives the location relative to the root
of the tree. When you put the parent in a larger tree, the child goes with it:

In [ ]:
root = Resource(name="root", size_x=500, size_y=300, size_z=0)
root.assign_child_resource(parent, location=Coordinate(x=100, y=50, z=0))

assert child.get_absolute_location() == Coordinate(x=110, y=70, z=0)

A name identifies a resource in its tree, so two resources in one tree cannot have the same name:

In [ ]:
duplicate = Resource(name="child", size_x=5, size_y=5, size_z=5)

try:
  parent.assign_child_resource(duplicate, location=Coordinate(x=0, y=0, z=0))
except ValueError as error:
  print(error)  # Resource with name 'child' already exists in the tree.
else:
  raise AssertionError("expected a ValueError")

{meth}`~pylabrobot.resources.Resource.unassign_child_resource` removes a child from its parent. The
child then has no parent and no location:

In [ ]:
parent.unassign_child_resource(child)

assert child.parent is None
assert child.location is None

## By anchor points: `assign_child_by_anchor`

An anchor is a point on a resource, given by one letter per axis:

- x: `"l"` (left), `"c"` (center) or `"r"` (right);
- y: `"f"` (front), `"c"` (center) or `"b"` (back);
- z: `"b"` (bottom), `"c"` (center) or `"t"` (top).

`get_anchor` and the tuple form also take the full words, such as `"left"`.
{meth}`~pylabrobot.resources.Resource.get_anchor` gives the location of an anchor, relative to the
left front bottom corner of the resource:

In [ ]:
assert parent.get_anchor(x="c", y="c", z="t") == Coordinate(x=50, y=40, z=10)

`assign_child_by_anchor` puts the child so that its `child_anchor` is at the `parent_anchor` of the
parent. Then it calls `assign_child_resource` with that location. Both anchors are left front
bottom (`"lfb"`) by default, which gives the location `(0, 0, 0)`.

Each of the next cells unassigns the child at its end, so that the next cell can assign it again.

To center the child on the parent, align the center of the bottom of both:

In [ ]:
parent.assign_child_by_anchor(child, parent_anchor="ccb", child_anchor="ccb")

# (100 - 20) / 2 = 40 and (80 - 10) / 2 = 35
assert child.location == Coordinate(x=40, y=35, z=0)
parent.unassign_child_resource(child)

To put the child on top of the parent, align the top of the parent with the bottom of the child.
The anchors can also be tuples:

In [ ]:
parent.assign_child_by_anchor(child, parent_anchor=("c", "c", "t"), child_anchor=("c", "c", "b"))
assert child.location == Coordinate(x=40, y=35, z=10)
parent.unassign_child_resource(child)

# In the right back corner, on top
parent.assign_child_by_anchor(child, parent_anchor="rbt", child_anchor="rbb")
assert child.location == Coordinate(x=80, y=70, z=10)
parent.unassign_child_resource(child)

## Resources that compute the location

Some resources know where their children go. For these, you do not give a location.

A {class}`~pylabrobot.resources.ResourceHolder` puts its child at its `child_location`:

In [ ]:
from pylabrobot.resources import ResourceHolder, cor_96_wellplate_360uL_Fb

holder = ResourceHolder(
  name="holder", size_x=127.76, size_y=85.48, size_z=0, child_location=Coordinate(x=0, y=0, z=2)
)
plate = cor_96_wellplate_360uL_Fb(name="plate")
holder.assign_child_resource(plate)

assert holder.resource is plate
assert plate.location == Coordinate(x=0, y=0, z=2)

A {class}`~pylabrobot.resources.Carrier` has sites, and each site is a resource holder. Index the
carrier to assign a resource to a site:

In [ ]:
from pylabrobot.resources import hamilton_plate_carrier_L5_ac

plate_carrier = hamilton_plate_carrier_L5_ac(name="plate_carrier")
plate_carrier[0] = cor_96_wellplate_360uL_Fb(name="plate_on_carrier")

assert plate_carrier[0].resource.name == "plate_on_carrier"
assert plate_carrier[0].resource.parent is plate_carrier.sites[0]

Decks have their own ways to place resources. A Hamilton deck takes a `track` and computes the
location of the carrier from it:

In [ ]:
from pylabrobot.resources import STARLetDeck

star_deck = STARLetDeck()
star_deck.assign_child_resource(plate_carrier, track=1)

assert plate_carrier.parent is star_deck

An Opentrons deck has a resource holder for each slot. Use
{meth}`~pylabrobot.resources.opentrons.deck.OTDeck.assign_child_at_slot` to put labware in a slot:

In [ ]:
from pylabrobot.resources import OTDeck

ot_deck = OTDeck()
ot_deck.assign_child_at_slot(cor_96_wellplate_360uL_Fb(name="plate_in_slot_1"), 1)

assert ot_deck.get_resource("plate_in_slot_1").parent.name == "ot2_deck_slot_1"

See {class}`~pylabrobot.resources.Resource` for the API reference.